# CS F407 - Artificial Intelligence
## Laboratory: Bayesian Networks and Autoregressive Language Models

**Student:** Eshan Saikia  
**ID:** 2023A7PS1024G

This notebook contains the executable probability models, conditional tables, generation experiments, model comparison, and validation evidence. The accompanying report contains the written answers and interpretation.

## 1. From Probability to Language

For a token sequence,

\[
P(x_1,\ldots,x_T)=P(x_1)\prod_{t=2}^{T}P(x_t\mid x_1,\ldots,x_{t-1}).
\]

This decomposition is useful for generation because text can be generated one token at a time: sample the first token, condition on the generated prefix, sample the next token, and continue until `<END>`.

For `the cat sat on the mat`, the chain rule separates the probability of the complete sequence into conditional next-token probabilities.

## 2. First-Order Bayesian Network

A first-order model uses the approximation

\[
P(X_t\mid X_1,\ldots,X_{t-1})\approx P(X_t\mid X_{t-1}).
\]

Its graph is

\[
X_1\rightarrow X_2\rightarrow\cdots\rightarrow X_T.
\]

The associated independence assumption is

\[
X_t\perp\{X_1,\ldots,X_{t-2}\}\mid X_{t-1}.
\]

Thus, once the immediately preceding token is known, earlier tokens are assumed not to add information for predicting the next token.

In [1]:
from collections import Counter, defaultdict
import random
import math

sentences = [
    "the cat sat on the mat",
    "the cat sat on the rug",
    "the dog sat on the mat",
    "the dog ran to the park",
    "the cat ran to the park",
    "the dog sat on the rug",
]

tokenised = [["<START>"] + sentence.lower().split() + ["<END>"] for sentence in sentences]

first_counts = defaultdict(Counter)
for sent in tokenised:
    for current, nxt in zip(sent[:-1], sent[1:]):
        first_counts[current][nxt] += 1

def normalize_counts(counts):
    return {
        context: {
            token: count / sum(counter.values())
            for token, count in counter.items()
        }
        for context, counter in counts.items()
    }

first_probs = normalize_counts(first_counts)
vocab = sorted({token for sent in tokenised for token in sent})

print("Tokenised training data:")
for sent in tokenised:
    print(sent)
print("\nVocabulary size:", len(vocab))
print("Observed first-order contexts:", len(first_probs))

Tokenised training data:
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'mat', '<END>']
['<START>', 'the', 'dog', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'cat', 'ran', 'to', 'the', 'park', '<END>']
['<START>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '<END>']

Vocabulary size: 12
Observed first-order contexts: 11


## 3. Conditional Probability Table

The first-order model estimates

\[
P(w_j\mid w_i)=\frac{C(w_i,w_j)}{\sum_k C(w_i,w_k)}.
\]

The required conditional distributions are displayed below. Only nonzero outcomes are printed; every unobserved vocabulary transition has probability zero.

In [2]:
for context in ["the", "cat", "dog", "sat", "ran"]:
    print(f"P(next | {context!r}):")
    for token, probability in first_probs[context].items():
        print(f"  {token:>8}: {probability:.6f}")
    print()

P(next | 'the'):
       cat: 0.250000
       mat: 0.166667
       rug: 0.166667
       dog: 0.250000
      park: 0.166667

P(next | 'cat'):
       sat: 0.666667
       ran: 0.333333

P(next | 'dog'):
       sat: 0.666667
       ran: 0.333333

P(next | 'sat'):
        on: 1.000000

P(next | 'ran'):
        to: 1.000000



### Zero-probability transitions

The model is unsmoothed, so every unobserved transition has probability 0. For example,

\[
P(\texttt{on}\mid\texttt{the})=0,\qquad
P(\texttt{the}\mid\texttt{cat})=0,\qquad
P(\texttt{sat}\mid\texttt{ran})=0.
\]

## 4. LLM-Assisted Implementation

### Exact first-order prompt

> Write a simple Python implementation of a first-order autoregressive language model. The model should: (1) take tokenised sentences as training data; (2) count transitions between consecutive tokens; (3) construct \(P(X_t\mid X_{t-1})\); (4) display probabilities for a specified previous token; (5) predict the most probable next token; (6) generate a sentence by repeatedly sampling the next token; and (7) stop at `<END>`. Do not use a machine-learning library or pretrained model. Use ordinary Python data structures and random sampling. Add a safe failure case for a context with no observed continuation.

The generated structure was inspected before execution. Validation checks and the second-order extension were then added while keeping the specified probabilistic model unchanged.

In [3]:
# Transition counts and conditional probabilities are kept in ordinary Python data structures.
print("Transition counts for 'the':", dict(first_counts["the"]))
print("Conditional distribution for 'the':", first_probs["the"])

Transition counts for 'the': {'cat': 3, 'mat': 2, 'rug': 2, 'dog': 3, 'park': 2}
Conditional distribution for 'the': {'cat': 0.25, 'mat': 0.16666666666666666, 'rug': 0.16666666666666666, 'dog': 0.25, 'park': 0.16666666666666666}


## 5. Model Validation and Next-Word Prediction

For every context with observed outgoing transitions, a correct conditional distribution must satisfy

\[
\sum_v P(v\mid w)=1.
\]

A row summing to 0.87 would indicate an invalid probability distribution and therefore a bug in the counting, denominator, or normalisation logic.

The most probable next token is

\[
\arg\max_v P(v\mid w).
\]

In [4]:
for context, distribution in first_probs.items():
    total = sum(distribution.values())
    print(f"{context:>8}: sum = {total:.12f}")
    assert math.isclose(total, 1.0, rel_tol=0.0, abs_tol=1e-12)

print("\nAll observed first-order rows are normalised.")
print("\nMost probable next token:")
for context in ["the", "cat", "dog", "sat", "ran"]:
    nxt = max(first_probs[context].items(), key=lambda item: item[1])[0]
    print(f"{context:>5} -> {nxt}")

 <START>: sum = 1.000000000000
     the: sum = 1.000000000000
     cat: sum = 1.000000000000
     sat: sum = 1.000000000000
      on: sum = 1.000000000000
     mat: sum = 1.000000000000
     rug: sum = 1.000000000000
     dog: sum = 1.000000000000
     ran: sum = 1.000000000000
      to: sum = 1.000000000000
    park: sum = 1.000000000000

All observed first-order rows are normalised.

Most probable next token:
  the -> cat
  cat -> sat
  dog -> sat
  sat -> on
  ran -> to


In [5]:
def choose_sample(distribution):
    return random.choices(list(distribution.keys()), weights=list(distribution.values()), k=1)[0]

def choose_greedy(distribution):
    return max(distribution.items(), key=lambda item: item[1])[0]

def generate_first_order(mode="sampling", max_tokens=40, detect_loop=False):
    current = "<START>"
    output = []
    seen_contexts = set()

    for _ in range(max_tokens):
        if current not in first_probs:
            return output, False, False
        if detect_loop and current in seen_contexts:
            return output, False, True
        seen_contexts.add(current)

        nxt = choose_greedy(first_probs[current]) if mode == "greedy" else choose_sample(first_probs[current])
        if nxt == "<END>":
            return output, True, False
        output.append(nxt)
        current = nxt

    return output, False, False

random.seed(20261001)
first_20 = [generate_first_order("sampling") for _ in range(20)]
random.seed(20261002)
first_sampling_5 = [generate_first_order("sampling") for _ in range(5)]
first_greedy_5 = [generate_first_order("greedy", max_tokens=12, detect_loop=True) for _ in range(5)]

print("Generation functions ready.")

Generation functions ready.


## 6. Generating Text

Sampling draws from the complete conditional distribution, while greedy generation always chooses the maximum-probability outcome.

The full set of 20 first-order sampling generations is saved in the output below.

In [6]:
print("20 first-order sampling generations:")
print()
for i, sentence in enumerate(first_20, start=1):
    text = " ".join(sentence[0])
    print(f"{i:2d}. {text}")

20 first-order sampling generations:

 1. the mat
 2. the mat
 3. the cat ran to the park
 4. the cat ran to the park
 5. the cat sat on the cat sat on the rug
 6. the cat ran to the rug
 7. the mat
 8. the park
 9. the dog ran to the park
10. the cat sat on the rug
11. the dog sat on the rug
12. the mat
13. the dog ran to the dog sat on the mat
14. the cat ran to the cat sat on the mat
15. the cat sat on the rug
16. the dog ran to the park
17. the dog sat on the park
18. the dog sat on the rug
19. the park
20. the mat


### Deterministic versus sampling generation

For this training set, first-order greedy generation exposes a deterministic cycle. The tie in the `the` row is resolved deterministically in favour of `cat`; the sequence then returns to `the` through `cat -> sat -> on`. A loop detector is used only to terminate the diagnostic run; it does not change the model or select a different token.

This means the required greedy experiment produces bounded traces rather than naturally terminated sentences, whereas sampling can choose alternative observed transitions and may reach `<END>`.

In [7]:
print("Five first-order greedy traces (LOOP means <END> was not reached):")
print()
for i, item in enumerate(first_greedy_5, start=1):
    suffix = " [LOOP DETECTED]" if item[2] else ""
    text = " ".join(item[0])
    print(f"{i}. {text}{suffix}")

print()
print("Five first-order sampling sentences:")
print()
for i, item in enumerate(first_sampling_5, start=1):
    text = " ".join(item[0])
    print(f"{i}. {text}")

Five first-order greedy traces (LOOP means <END> was not reached):

1. the cat sat on the [LOOP DETECTED]
2. the cat sat on the [LOOP DETECTED]
3. the cat sat on the [LOOP DETECTED]
4. the cat sat on the [LOOP DETECTED]
5. the cat sat on the [LOOP DETECTED]

Five first-order sampling sentences:

1. the cat ran to the park
2. the mat
3. the park
4. the park
5. the mat


**Question 10:** Sampling produces more variation because it can choose any observed continuation with nonzero probability. Greedy generation always selects the same maximum-probability continuation for a given context, so it is deterministic and can become trapped in a cycle. The first-order sampling outputs therefore vary, although they can also contain unusual combinations because only one-token context is retained.

## 7. Second-Order Bayesian Network

The second-order model estimates

\[
P(X_t\mid X_{t-2},X_{t-1}).
\]

The local graph is

\[
X_{t-2}\rightarrow X_t\leftarrow X_{t-1}.
\]

For four tokens,

\[
P(X_1,X_2,X_3,X_4)=P(X_1)P(X_2\mid X_1)P(X_3\mid X_1,X_2)P(X_4\mid X_2,X_3).
\]

Compared with the first-order model, the CPT is indexed by ordered token pairs, giving more context but requiring more data to populate the table reliably.

### Exact second-order modification prompt

> Modify the existing first-order autoregressive model into a second-order model estimating \(P(X_t\mid X_{t-2},X_{t-1})\). Count observed triples and construct conditional distributions indexed by the previous two tokens. Keep the same training sentences and special tokens. Support next-token prediction and sampling-based generation. Do not replace the model with a neural network or pretrained language model.

In [8]:
second_counts = defaultdict(Counter)
for sent in tokenised:
    for a, b, c in zip(sent[:-2], sent[1:-1], sent[2:]):
        second_counts[(a, b)][c] += 1

second_probs = normalize_counts(second_counts)

print("Observed second-order contexts:", len(second_probs))
print("P(next | 'the', 'cat'):", second_probs[("the", "cat")])
print("P(next | 'on', 'the'):", second_probs[("on", "the")])

Observed second-order contexts: 14
P(next | 'the', 'cat'): {'sat': 0.6666666666666666, 'ran': 0.3333333333333333}
P(next | 'on', 'the'): {'mat': 0.5, 'rug': 0.5}


In [9]:
def generate_second_order(mode="sampling", max_tokens=40, detect_loop=False):
    first_distribution = first_probs["<START>"]
    first = choose_greedy(first_distribution) if mode == "greedy" else choose_sample(first_distribution)
    if first == "<END>":
        return [], True, False

    output = [first]
    pair = ("<START>", first)
    seen_pairs = set()

    for _ in range(max_tokens - 1):
        if pair not in second_probs:
            return output, False, False
        if detect_loop and pair in seen_pairs:
            return output, False, True
        seen_pairs.add(pair)

        nxt = choose_greedy(second_probs[pair]) if mode == "greedy" else choose_sample(second_probs[pair])
        if nxt == "<END>":
            return output, True, False
        output.append(nxt)
        pair = (pair[1], nxt)

    return output, False, False

random.seed(20261003)
second_20 = [generate_second_order("sampling") for _ in range(20)]
random.seed(20261004)
second_sampling_5 = [generate_second_order("sampling") for _ in range(5)]
second_greedy_5 = [generate_second_order("greedy", max_tokens=12, detect_loop=True) for _ in range(5)]

print("Generation datasets prepared.")

Generation datasets prepared.


## 8. First-Order versus Second-Order Comparison

With 12 vocabulary tokens, the observed context tables contain:

| Measure | First-order | Second-order |
|---|---:|---:|
| Observed context rows | 11 | 14 |
| CPT cells over the vocabulary | 132 | 168 |
| Nonzero context-to-token entries | 17 | 18 |
| Possible context combinations | 12 | 144 |
| Unobserved possible contexts | 1 | 130 |

The second-order model is therefore larger and much sparser when all possible token-pair contexts are considered. Increasing context can improve prediction because it distinguishes situations that share the same immediately preceding word, but it simultaneously makes parameter estimation harder on a small dataset.

In [10]:
print(f"First-order:  {len(first_probs)} observed contexts, {len(first_probs) * len(vocab)} CPT cells, {sum(len(c) for c in first_counts.values())} nonzero entries")
print(f"Second-order: {len(second_probs)} observed contexts, {len(second_probs) * len(vocab)} CPT cells, {sum(len(c) for c in second_counts.values())} nonzero entries")
print(f"Possible contexts: first-order={len(vocab)}, second-order={len(vocab) ** 2}")
print(f"Unobserved contexts: first-order={len(vocab)-len(first_probs)}, second-order={len(vocab) ** 2-len(second_probs)}")

First-order:  11 observed contexts, 132 CPT cells, 17 nonzero entries
Second-order: 14 observed contexts, 168 CPT cells, 18 nonzero entries
Possible contexts: first-order=12, second-order=144
Unobserved contexts: first-order=1, second-order=130


### Generated-text comparison

The 20 fixed-seed first-order sampling runs produced 8 unique outputs. The second-order runs produced 5 unique outputs, corresponding closely to the five sentence patterns present in the training set. The second-order model is less able to splice together transitions from unrelated contexts because two previous tokens are retained.

In [11]:
print("20 second-order sampling generations:")
print()
for i, sentence in enumerate(second_20, start=1):
    text = " ".join(sentence[0])
    print(f"{i:2d}. {text}")

first_unique_runtime = len(set(" ".join(x[0]) for x in first_20))
second_unique_runtime = len(set(" ".join(x[0]) for x in second_20))
print()
print(f"Unique first-order outputs: {first_unique_runtime}")
print(f"Unique second-order outputs: {second_unique_runtime}")

20 second-order sampling generations:

 1. the dog sat on the rug
 2. the cat sat on the rug
 3. the dog ran to the park
 4. the cat sat on the mat
 5. the dog sat on the mat
 6. the cat sat on the mat
 7. the dog sat on the mat
 8. the cat ran to the park
 9. the dog ran to the park
10. the dog sat on the mat
11. the cat sat on the mat
12. the dog ran to the park
13. the dog sat on the rug
14. the dog sat on the rug
15. the dog sat on the mat
16. the cat ran to the park
17. the dog ran to the park
18. the dog sat on the rug
19. the cat sat on the mat
20. the dog ran to the park

Unique first-order outputs: 11
Unique second-order outputs: 6


In [12]:
print("Five second-order greedy generations:")
print()
for i, item in enumerate(second_greedy_5, start=1):
    suffix = " [LOOP DETECTED]" if item[2] else ""
    text = " ".join(item[0])
    print(f"{i}. {text}{suffix}")

print()
print("Five second-order sampling generations:")
print()
for i, item in enumerate(second_sampling_5, start=1):
    text = " ".join(item[0])
    print(f"{i}. {text}")

Five second-order greedy generations:

1. the cat sat on the mat
2. the cat sat on the mat
3. the cat sat on the mat
4. the cat sat on the mat
5. the cat sat on the mat

Five second-order sampling generations:

1. the dog sat on the rug
2. the cat sat on the mat
3. the dog ran to the park
4. the dog ran to the park
5. the cat sat on the mat


## 9. Modern Autoregressive Language Models

A modern autoregressive language model also models

\[
P(X_t\mid X_1,\ldots,X_{t-1}).
\]

The small models here estimate conditional distributions with explicit CPTs. Modern systems use neural networks to estimate the conditional distribution over a very large vocabulary. The shared probabilistic idea is next-token prediction and chain-rule factorisation; the representation, scale, learned parameters, context handling, and training procedure are very different.

## 10. LLM Reflection

A behavioural specification is preferable to “write a language model” because it fixes the intended variables, dependencies, probability distribution, and generation behaviour before code is generated.

The LLM provided a starting implementation for transition counting, conditional probabilities, and generation. Independent validation was needed to confirm the CPT sums, inspect zero-probability transitions, verify generation behaviour, and check that the second-order implementation really conditioned on two previous tokens.

Approach B is preferable because the probabilistic model is specified before implementation. This makes the representation testable and separates the intended model from the code used to implement it.

## 11. What the Bayesian-Network View Adds

Thinking of the language model as a Bayesian network provides an explicit dependency structure, a factorisation of the joint distribution, an interpretation of conditional probabilities, a principled sequential-generation procedure, a precise independence assumption for the first-order model, and a clear way to reason about the effect of increasing context and data sparsity.

The overall progression is

\[
\text{Probability}\rightarrow\text{Bayesian Network}\rightarrow\text{Autoregressive Model}\rightarrow\text{Language Generation}.
\]

## 12. Verification Checklist

- [x] Questions 1-14 addressed.
- [x] First-order conditional probability tables constructed.
- [x] Zero-probability transitions identified.
- [x] Probability normalisation tested.
- [x] Next-word predictions tested.
- [x] 20 first-order generated sentences saved.
- [x] Greedy and sampling modes tested.
- [x] Second-order model implemented from triple counts.
- [x] First-order and second-order model sizes compared.
- [x] Generation diversity and qualitative differences compared.
- [x] LLM prompts and validation process recorded.
- [x] No machine-learning library or pretrained model used.